# Создание golden set: отдельный CSV на каждый класс

Ноутбук объединяет два размеченных CSV-файла, разбирает multi-label разметку и для каждого класса сохраняет отдельный CSV:

- если по классу есть **100 или больше** примеров — случайно выбирает 100;
- если по классу есть **меньше 100** примеров — сохраняет все доступные примеры;
- один и тот же отзыв может попасть в несколько файлов, если у него несколько labels.

Результат сохраняется в отдельную папку `golden_set_by_class`.

In [10]:
# Если запускаешь в Google Colab, раскомментируй эту ячейку.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Импорты и настройки

In [11]:
from pathlib import Path
import re
import json

import numpy as np
import pandas as pd

In [12]:
# =========================
# ОСНОВНЫЕ НАСТРОЙКИ
# =========================

RANDOM_STATE = 42
SAMPLES_PER_CLASS = 100

# Если True, дубликаты строк удаляются по тексту отзыва и labels.
# Это защищает от случайного повторного попадания одинаковых строк из разных файлов.
DROP_DUPLICATES = True

# Если True, порядок строк внутри каждого class CSV будет случайным.
SHUFFLE_OUTPUT = True

# Разрешенные классы. Названия должны совпадать с разметкой.
ALLOWED_LABELS = [
    "Брак / дефект товара",
    "Низкое качество материала",
    "Проблема с размером / посадкой",
    "Несоответствие описанию",
    "Несоответствие ожиданиям / эффекту",
    "Проблема с комплектацией",
    "Проблема с упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Подделка / оригинальность",
    "Положительный отзыв",
    "Нейтральный / информационный отзыв",
    "Другая проблема",
]

## 2. Пути к входным и выходным файлам

По умолчанию указаны пути для твоей структуры на Google Drive. Если файлы лежат рядом с ноутбуком, можно заменить пути вручную.

In [13]:
# =========================
# ВАРИАНТ ДЛЯ GOOGLE DRIVE
# =========================

BASE_DIR = Path("/content/drive/MyDrive/MLops_project/project")
LABELED_DIR = BASE_DIR / "data" / "labeled"

INPUT_CSV_PATHS = [
    LABELED_DIR / "wb_feedbacks_llm_labeled_from_sample" / "balanced_50_per_class_random_chunks_final.csv",
    LABELED_DIR / "wb_feedbacks_llm_labeled_from_clusters" / "extra_from_hdbscan_candidate_clusters_final.csv",
]

OUTPUT_DIR = LABELED_DIR / "golden_set_by_class"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Входные файлы:")
for path in INPUT_CSV_PATHS:
    print(" -", path)

print("Выходная папка:")
print(OUTPUT_DIR)

Входные файлы:
 - /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_sample/balanced_50_per_class_random_chunks_final.csv
 - /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_clusters/extra_from_hdbscan_candidate_clusters_final.csv
Выходная папка:
/content/drive/MyDrive/MLops_project/project/data/labeled/golden_set_by_class


In [14]:
# =========================
# РЕЗЕРВНЫЙ ВАРИАНТ
# =========================
# Если ноутбук запускается локально или файлы лежат рядом с ним,
# можно раскомментировать и использовать такие пути:

# INPUT_CSV_PATHS = [
#     Path("balanced_50_per_class_random_chunks_final.csv"),
#     Path("extra_from_hdbscan_candidate_clusters_final.csv"),
# ]
# OUTPUT_DIR = Path("golden_set_by_class")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Загрузка данных

In [15]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    """Читает CSV и добавляет колонку source_file."""
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

    df = pd.read_csv(path)
    df["source_file"] = path.name
    return df

frames = []
for path in INPUT_CSV_PATHS:
    df_part = read_csv_safely(path)
    frames.append(df_part)
    print(f"Прочитан файл: {path.name}, shape={df_part.shape}")

df = pd.concat(frames, ignore_index=True)
print("Объединенный датасет:", df.shape)
df.head()

Прочитан файл: balanced_50_per_class_random_chunks_final.csv, shape=(146, 4)
Прочитан файл: extra_from_hdbscan_candidate_clusters_final.csv, shape=(866, 4)
Объединенный датасет: (1012, 4)


,отзыв,labels,evidence,source_file
0,"Качество хорошее, но мне влики, отказ",Другая проблема,"В отзыве упоминается 'влики, отказ', что указы...",balanced_50_per_class_random_chunks_final.csv
1,"Волшебно! Все супер, берите, не пожалеете)",Положительный отзыв,"В отзыве использованы слова 'Волшебно!', 'Все ...",balanced_50_per_class_random_chunks_final.csv
2,❤️❤️❤️❤️❤️ просто прелесть,Положительный отзыв,Использованы эмоциональные символы ❤️ и выраже...,balanced_50_per_class_random_chunks_final.csv
3,Все пучком,Положительный отзыв,Фраза 'Все пучком' является прямой положительн...,balanced_50_per_class_random_chunks_final.csv
4,Очень вкусноооо))) Спасибо большое!,Положительный отзыв,Покупатель пишет 'очень вкусноооо' и 'спасибо ...,balanced_50_per_class_random_chunks_final.csv


In [16]:
# Проверка обязательных колонок
required_columns = {"отзыв", "labels"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Не хватает обязательных колонок: {missing_columns}")

# Приводим текстовые поля к строкам
for col in ["отзыв", "labels", "evidence"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

if DROP_DUPLICATES:
    before = len(df)
    df = df.drop_duplicates(subset=["отзыв", "labels"]).reset_index(drop=True)
    after = len(df)
    print(f"Удалено дубликатов: {before - after}")

print("Итоговый размер после подготовки:", df.shape)

Удалено дубликатов: 0
Итоговый размер после подготовки: (1012, 4)


## 4. Разбор multi-label разметки

В твоих CSV несколько классов в одной строке разделяются через `;`. Эта ячейка превращает строку labels в список классов.

In [17]:
def parse_labels(value: str) -> list[str]:
    """Парсит строку labels в список валидных классов."""
    if pd.isna(value):
        return []

    value = str(value).strip()
    if not value:
        return []

    # Основной формат: label1; label2; label3
    parts = [part.strip() for part in value.split(";")]
    parts = [part for part in parts if part]

    # Оставляем только разрешенные классы, сохраняя порядок и убирая повторы
    seen = set()
    parsed = []
    for part in parts:
        if part in ALLOWED_LABELS and part not in seen:
            parsed.append(part)
            seen.add(part)

    return parsed


df["parsed_labels"] = df["labels"].apply(parse_labels)
df["num_labels"] = df["parsed_labels"].apply(len)

bad_rows = df[df["num_labels"] == 0]
print("Строк без валидных labels:", len(bad_rows))

if len(bad_rows) > 0:
    display(bad_rows[["отзыв", "labels", "source_file"]].head(20))

df_valid = df[df["num_labels"] > 0].copy().reset_index(drop=True)
print("Строк с валидными labels:", len(df_valid))

Строк без валидных labels: 0
Строк с валидными labels: 1012


## 5. Текущее количество примеров по классам

In [18]:
class_counts = {
    label: int(df_valid["parsed_labels"].apply(lambda labels: label in labels).sum())
    for label in ALLOWED_LABELS
}

counts_df = pd.DataFrame({
    "class": list(class_counts.keys()),
    "available_examples": list(class_counts.values()),
})
counts_df["selected_for_golden_set"] = counts_df["available_examples"].clip(upper=SAMPLES_PER_CLASS)
counts_df["missing_to_100"] = (SAMPLES_PER_CLASS - counts_df["available_examples"]).clip(lower=0)
counts_df = counts_df.sort_values("available_examples", ascending=False).reset_index(drop=True)

display(counts_df)

summary_path = OUTPUT_DIR / "golden_set_class_counts_summary.csv"
counts_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("Сводка сохранена:", summary_path)

,class,available_examples,selected_for_golden_set,missing_to_100
0,Проблема с упаковкой,318,100,0
1,Брак / дефект товара,215,100,0
2,Несоответствие описанию,175,100,0
3,Проблема с комплектацией,171,100,0
4,Проблема доставки / получения,140,100,0
5,Положительный отзыв,135,100,0
6,Несоответствие ожиданиям / эффекту,125,100,0
7,Цена / ценность,78,78,22
8,Другая проблема,64,64,36
9,Низкое качество материала,56,56,44


Сводка сохранена: /content/drive/MyDrive/MLops_project/project/data/labeled/golden_set_by_class/golden_set_class_counts_summary.csv


## 6. Сохранение отдельного CSV для каждого класса

Файлы будут называться безопасно, без `/`, чтобы нормально сохраняться на диск.

In [20]:
def safe_filename(label: str) -> str:
    """Делает безопасное имя файла из названия класса."""
    name = label.lower()
    name = name.replace("/", "_")
    name = re.sub(r"[^а-яА-Яa-zA-Z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def sample_class_examples(df_source: pd.DataFrame, label: str, n: int, random_state: int) -> pd.DataFrame:
    """Выбирает до n примеров, где среди labels есть нужный класс."""
    class_df = df_source[df_source["parsed_labels"].apply(lambda labels: label in labels)].copy()

    if len(class_df) > n:
        class_df = class_df.sample(n=n, random_state=random_state)
    elif SHUFFLE_OUTPUT and len(class_df) > 0:
        class_df = class_df.sample(frac=1.0, random_state=random_state)

    class_df = class_df.reset_index(drop=True)
    class_df.insert(0, "target_class", label)
    class_df.insert(1, "golden_set_sample_size", len(class_df))
    class_df.insert(2, "available_examples_for_class", int(class_counts[label]))

    return class_df

saved_files = []

for idx, label in enumerate(ALLOWED_LABELS, start=1):
    sampled_df = sample_class_examples(
        df_source=df_valid,
        label=label,
        n=SAMPLES_PER_CLASS,
        random_state=RANDOM_STATE + idx,
    )

    filename = f"{idx:02d}_{safe_filename(label)}.csv"
    output_path = OUTPUT_DIR / filename

    # parsed_labels — список, поэтому перед сохранением сделаем его строкой для удобства CSV
    sampled_to_save = sampled_df.copy()
    sampled_to_save["parsed_labels"] = sampled_to_save["parsed_labels"].apply(lambda labels: "; ".join(labels))

    sampled_to_save.to_csv(output_path, index=False, encoding="utf-8-sig")

    saved_files.append({
        "class_order": idx,
        "class": label,
        "available_examples": int(class_counts[label]),
        "saved_examples": int(len(sampled_to_save)),
        "missing_to_100": int(max(0, SAMPLES_PER_CLASS - class_counts[label])),
        "file": str(output_path),
    })

saved_files_df = pd.DataFrame(saved_files)
display(saved_files_df)

saved_files_summary_path = OUTPUT_DIR / "golden_set_saved_files_summary.csv"
saved_files_df.to_csv(saved_files_summary_path, index=False, encoding="utf-8-sig")

print("Готово. Файлы сохранены в папку:")
print(OUTPUT_DIR)
print("Сводка по сохраненным файлам:")
print(saved_files_summary_path)

,class_order,class,available_examples,saved_examples,missing_to_100,file
0,1,Брак / дефект товара,215,100,0,/content/drive/MyDrive/MLops_project/project/d...
1,2,Низкое качество материала,56,56,44,/content/drive/MyDrive/MLops_project/project/d...
2,3,Проблема с размером / посадкой,54,54,46,/content/drive/MyDrive/MLops_project/project/d...
3,4,Несоответствие описанию,175,100,0,/content/drive/MyDrive/MLops_project/project/d...
4,5,Несоответствие ожиданиям / эффекту,125,100,0,/content/drive/MyDrive/MLops_project/project/d...
5,6,Проблема с комплектацией,171,100,0,/content/drive/MyDrive/MLops_project/project/d...
6,7,Проблема с упаковкой,318,100,0,/content/drive/MyDrive/MLops_project/project/d...
7,8,Проблема доставки / получения,140,100,0,/content/drive/MyDrive/MLops_project/project/d...
8,9,Цена / ценность,78,78,22,/content/drive/MyDrive/MLops_project/project/d...
9,10,Подделка / оригинальность,18,18,82,/content/drive/MyDrive/MLops_project/project/d...


Готово. Файлы сохранены в папку:
/content/drive/MyDrive/MLops_project/project/data/labeled/golden_set_by_class
Сводка по сохраненным файлам:
/content/drive/MyDrive/MLops_project/project/data/labeled/golden_set_by_class/golden_set_saved_files_summary.csv


## 7. Дополнительно: общий файл со всеми выбранными примерами

Этот файл удобен, если потом захочешь смотреть весь golden set в одном месте. В нем будут повторы строк, если один отзыв попал в несколько классов.

In [21]:
all_selected = []

for idx, label in enumerate(ALLOWED_LABELS, start=1):
    sampled_df = sample_class_examples(
        df_source=df_valid,
        label=label,
        n=SAMPLES_PER_CLASS,
        random_state=RANDOM_STATE + idx,
    )
    sampled_df["class_order"] = idx
    all_selected.append(sampled_df)

all_selected_df = pd.concat(all_selected, ignore_index=True)
all_selected_df["parsed_labels"] = all_selected_df["parsed_labels"].apply(lambda labels: "; ".join(labels))

all_selected_path = OUTPUT_DIR / "golden_set_all_classes_stacked.csv"
all_selected_df.to_csv(all_selected_path, index=False, encoding="utf-8-sig")

print("Общий файл сохранен:", all_selected_path)
print("Размер общего файла:", all_selected_df.shape)

display(all_selected_df.head())

Общий файл сохранен: /content/drive/MyDrive/MLops_project/project/data/labeled/golden_set_by_class/golden_set_all_classes_stacked.csv
Размер общего файла: (984, 10)


,target_class,golden_set_sample_size,available_examples_for_class,отзыв,labels,evidence,source_file,parsed_labels,num_labels,class_order
0,Брак / дефект товара,100,215,Бокалы пришли в рванной коробке и один бокал с...,Брак / дефект товара; Низкое качество материал...,"Упомянут брак (откололся край), качество на тр...",extra_from_hdbscan_candidate_clusters_final.csv,Брак / дефект товара; Низкое качество материал...,3,1
1,Брак / дефект товара,100,215,"Маловероятно, ещё и с затяжками. Не захотели в...",Брак / дефект товара,Упоминание о браке и необходимости оформления ...,extra_from_hdbscan_candidate_clusters_final.csv,Брак / дефект товара,1,1
2,Брак / дефект товара,100,215,Заказ шёл долго. Катушки сломанные пришли. Нет...,Проблема доставки / получения; Брак / дефект т...,Заказ шёл долго — проблема доставки; катушки с...,balanced_50_per_class_random_chunks_final.csv,Проблема доставки / получения; Брак / дефект т...,4,1
3,Брак / дефект товара,100,215,"Пришёл товар разбитый, это вообще что? Знают ч...",Брак / дефект товара; Проблема с упаковкой,"Товар пришел разбитым, что указывает на брак и...",extra_from_hdbscan_candidate_clusters_final.csv,Брак / дефект товара; Проблема с упаковкой,2,1
4,Брак / дефект товара,100,215,Никакой фирменной упаковки. Пришло в обычном п...,Проблема с упаковкой; Подделка / оригинальност...,"Упаковка — обычный полиэтиленовый пакет, а не ...",extra_from_hdbscan_candidate_clusters_final.csv,Проблема с упаковкой; Подделка / оригинальност...,3,1


## 8. Проверка результата

In [22]:
print("Файлы в выходной папке:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path.name)

Файлы в выходной папке:
- 01_брак_дефект_товара.csv
- 02_низкое_качество_материала.csv
- 03_проблема_с_размером_посадкой.csv
- 04_несоответствие_описанию.csv
- 05_несоответствие_ожиданиям_эффекту.csv
- 06_проблема_с_комплектацией.csv
- 07_проблема_с_упаковкой.csv
- 08_проблема_доставки_получения.csv
- 09_цена_ценность.csv
- 10_подделка_оригинальность.csv
- 11_положительный_отзыв.csv
- 12_нейтральный_информационный_отзыв.csv
- 13_другая_проблема.csv
- golden_set_all_classes_stacked.csv
- golden_set_class_counts_summary.csv
- golden_set_saved_files_summary.csv
